In [ ]:
# Rode primeiro: coloca o projeto no `sys.path` para importar `src.*`.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Análise rápida: 1 review de 1 pilar (tradução → PyABSA → GPT). Lote completo: `scripts/run_shein_absa.py`.
import json

import pandas as pd
from IPython.display import display

from src.processing.shein_absa_pipeline import run_shein_absa_for_row
from src.utils.helpers import reviews_data_dir
from src.utils.shein_reviews import sample_reviews_per_pillar

In [ ]:


# --- ajuste aqui ---
# Pilares: qualidade_produto | ajuste_caimento | logistica_entrega | atendimento_cliente | general_collection
PILAR = "qualidade_produto"
SEED = 42
# JSON com pillars em data/reviews/ (ex.: shein_small.json ou shein.json)
REVIEWS_JSON = reviews_data_dir() / "shein_small.json"

by_pillar = sample_reviews_per_pillar(1, seed=SEED, path=REVIEWS_JSON)
bucket = by_pillar.get(PILAR) or []
if not bucket:
    raise ValueError(
        f"Sem reviews para o pilar {PILAR!r}. Tente outro PILAR ou verifique {REVIEWS_JSON}."
    )

r = bucket[0]
t = (r.get("title") or "").strip()
body = (r.get("review") or "").strip()

print(f"Pilar: {PILAR}  |  label: {r.get('pillar_label', '')}\n")
print("=== PT-BR ===")
print(t)
print(body)

texto_en, result = run_shein_absa_for_row(r)
print("\n=== EN (ChatGPT) ===")
print(texto_en)

print("\n=== JSON ===")
print(json.dumps(result, ensure_ascii=False, indent=2))

asp = result.get("aspectos") or []
if asp:
    print("\n=== Aspectos (tabela) ===")
    display(pd.DataFrame(asp))

In [ ]:
# 1 — scrape → data/reviews/
uv run python -m src.scraping.scraper -c src/config/scraping_config_small.json -i shein_small.json -o shein_small.json
# 2 — ABSA → data/processed/
uv run python scripts/run_shein_absa.py -i shein_small.json -o shein_absa_small.json
